<a href="https://colab.research.google.com/github/Nikhitha158/hardware-trojan-detection/blob/main/finalproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

GITHUB_USERNAME = 'Nikhitha158'
REPO_NAME       = 'hardware-trojan-detection'

# Use HTTPS without credentials for public repo
!git clone https://github.com/Nikhitha158/hardware-trojan-detection.git /content/hardware-trojan-detection

os.chdir('/content/hardware-trojan-detection')
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir()}')

import tensorflow as tf
print(f'\nTensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

fatal: destination path '/content/hardware-trojan-detection' already exists and is not an empty directory.
Working directory: /content/hardware-trojan-detection
Files: ['results', '.git', 'README.md', 'models']

TensorFlow : 2.19.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
!git config --global user.email "palakurla.2@wright.edu"
!git config --global user.name "Nikhitha158"
print('Git configured.')

Git configured.


In [ ]:
import zipfile
import os

zips = [
    "/content/AES-T400_power_Temp25C.zip",
    "/content/AES-T500_power_Temp25C.zip",
    "/content/AES-T600_power_Temp25C.zip",
    "/content/AES-T700_power_Temp25C.zip"
]

base_dir = "/content/aes_dataset"

os.makedirs(base_dir, exist_ok=True)

for z in zips:
    with zipfile.ZipFile(z, 'r') as zip_ref:
        zip_ref.extractall(base_dir)

print("Extraction done")

In [ ]:
import numpy as np
import os

def load_all_datasets(base_path):
    data = []
    labels = []
    domains = []  # temperature label

    for root, _, files in os.walk(base_path):
        for f in files:
            path = os.path.join(root, f)

            if f.endswith(".npy"):
                x = np.load(path)
            elif f.endswith(".csv"):
                x = np.loadtxt(path, delimiter=",")
            elif f.endswith(".txt"):
                try:
                    x = np.loadtxt(path)
                except:
                    continue
            else:
                continue

            # label inference (we refine later if metadata exists)
            if "trojan" in f.lower():
                y = 1
            else:
                y = 0

            # domain extraction (T400, T500, etc.)
            if "T400" in path:
                d = 400
            elif "T500" in path:
                d = 500
            elif "T600" in path:
                d = 600
            elif "T700" in path:
                d = 700
            else:
                d = -1

            data.append(x)
            labels.append(y)
            domains.append(d)

    return np.array(data, dtype=object), np.array(labels), np.array(domains)


data, labels, domains = load_all_datasets(base_dir)

print("Samples:", len(data))
print("Labels:", np.bincount(labels))
print("Domains:", np.unique(domains))

In [ ]:
def window_trace(trace, window=200, step=50):
    windows = []
    for i in range(0, len(trace) - window, step):
        windows.append(trace[i:i+window])
    return windows

In [ ]:
import random
import torch

class AESPairDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels, domains, num_pairs=80000):
        self.data = data
        self.labels = labels
        self.domains = domains
        self.num_pairs = num_pairs

        self.clean_idx = np.where(labels == 0)[0]
        self.trojan_idx = np.where(labels == 1)[0]

    def __len__(self):
        return self.num_pairs

    def __getitem__(self, idx):

        # 50% similar, 50% dissimilar
        if random.random() < 0.5:
            y = 1
            cls = random.choice([0, 1])
            idx1, idx2 = random.sample(list(np.where(self.labels == cls)[0]), 2)
        else:
            y = 0
            idx1 = random.choice(self.clean_idx)
            idx2 = random.choice(self.trojan_idx)

        x1 = self.data[idx1]
        x2 = self.data[idx2]

        # normalization
        x1 = (x1 - np.mean(x1)) / (np.std(x1) + 1e-8)
        x2 = (x2 - np.mean(x2)) / (np.std(x2) + 1e-8)

        x1 = torch.tensor(x1, dtype=torch.float32).unsqueeze(0)
        x2 = torch.tensor(x2, dtype=torch.float32).unsqueeze(0)

        return x1, x2, torch.tensor(y, dtype=torch.float32)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SiameseAES(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, 7),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, 5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 3),
            nn.ReLU()
        )

        self.attn = nn.Sequential(
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.Softmax(dim=1)
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

    def forward_once(self, x):
        x = self.cnn(x)
        x = x.mean(dim=2)

        # attention
        w = self.attn(x)
        x = x * w

        return self.fc(x)

    def forward(self, x1, x2):
        return self.forward_once(x1), self.forward_once(x2)

In [ ]:
import torch.nn.functional as F

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, out1, out2, y):
        dist = F.pairwise_distance(out1, out2)

        loss = (1 - y) * dist**2 + y * torch.clamp(self.margin - dist, min=0)**2
        return loss.mean()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SiameseAES().to(device)
criterion = ContrastiveLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

dataset = AESPairDataset(data, labels, domains)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(10):
    model.train()
    total = 0

    for x1, x2, y in loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)

        optimizer.zero_grad()
        o1, o2 = model(x1, x2)

        loss = criterion(o1, o2, y)
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch+1} Loss: {total/len(loader):.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate(model, loader):
    model.eval()

    y_true, y_scores = [], []

    with torch.no_grad():
        for x1, x2, y in loader:
            x1, x2 = x1.to(device), x2.to(device)

            o1, o2 = model(x1, x2)
            dist = F.pairwise_distance(o1, o2)

            score = torch.sigmoid(-dist).cpu().numpy()

            y_scores.extend(score)
            y_true.extend(y.numpy())

    print("ROC-AUC:", roc_auc_score(y_true, y_scores))

In [ ]:
import random
import torch
import numpy as np

class AES_TripletDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels, num_samples=100000):
        self.data = data
        self.labels = labels
        self.num_samples = num_samples

        self.clean_idx = np.where(labels == 0)[0]
        self.trojan_idx = np.where(labels == 1)[0]

    def __len__(self):
        return self.num_samples

    def normalize(self, x):
        return (x - np.mean(x)) / (np.std(x) + 1e-8)

    def __getitem__(self, idx):

        # Anchor
        a_idx = random.choice(self.clean_idx)
        anchor = self.data[a_idx]

        # Positive (same class)
        p_idx = random.choice(self.clean_idx)
        positive = self.data[p_idx]

        # Negative (different class)
        n_idx = random.choice(self.trojan_idx)
        negative = self.data[n_idx]

        anchor = torch.tensor(self.normalize(anchor), dtype=torch.float32).unsqueeze(0)
        positive = torch.tensor(self.normalize(positive), dtype=torch.float32).unsqueeze(0)
        negative = torch.tensor(self.normalize(negative), dtype=torch.float32).unsqueeze(0)

        return anchor, positive, negative

In [ ]:
import torch.nn.functional as F

class TripletLoss(torch.nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, a, p, n):
        d_pos = F.pairwise_distance(a, p)
        d_neg = F.pairwise_distance(a, n)

        loss = torch.clamp(d_pos - d_neg + self.margin, min=0.0)
        return loss.mean()

In [ ]:
import torch
import torch.nn as nn

class AES_Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, 7),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, 5),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 3),
            nn.ReLU()
        )

        # Attention layer (important for AES leakage points)
        self.attn = nn.Linear(128, 128)

        # Projection head (VERY IMPORTANT for contrastive learning quality)
        self.proj = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.mean(dim=2)

        w = torch.sigmoid(self.attn(x))
        x = x * w

        return self.proj(x)

In [ ]:
class TripletNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AES_Encoder()

    def forward(self, a, p, n):
        return (
            self.encoder(a),
            self.encoder(p),
            self.encoder(n)
        )

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TripletNet().to(device)
criterion = TripletLoss(margin=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

dataset = AES_TripletDataset(data, labels)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(15):
    model.train()
    total_loss = 0

    for a, p, n in loader:
        a, p, n = a.to(device), p.to(device), n.to(device)

        optimizer.zero_grad()

        a_out, p_out, n_out = model(a, p, n)

        loss = criterion(a_out, p_out, n_out)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss {total_loss/len(loader):.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
import torch.nn.functional as F

def evaluate(model, data, labels):
    model.eval()

    scores = []
    y_true = []

    with torch.no_grad():
        for i in range(5000):

            a_idx = np.random.randint(0, len(data))
            b_idx = np.random.randint(0, len(data))

            a = torch.tensor(data[a_idx], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            b = torch.tensor(data[b_idx], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

            a = (a - a.mean()) / (a.std() + 1e-8)
            b = (b - b.mean()) / (b.std() + 1e-8)

            emb1 = model.encoder(a)
            emb2 = model.encoder(b)

            dist = F.pairwise_distance(emb1, emb2)

            score = torch.sigmoid(-dist).item()

            scores.append(score)
            y_true.append(int(labels[a_idx] == labels[b_idx]))

    auc = roc_auc_score(y_true, scores)

    print("ROC-AUC:", auc)

In [ ]:
def compute_far_frr(model, data, labels, threshold=0.5):
    model.eval()

    far = 0
    frr = 0
    total_far = 0
    total_frr = 0

    with torch.no_grad():
        for i in range(2000):
            a, b = np.random.randint(0, len(data), 2)

            x1 = torch.tensor(data[a], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            x2 = torch.tensor(data[b], dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

            x1 = (x1 - x1.mean()) / (x1.std() + 1e-8)
            x2 = (x2 - x2.mean()) / (x2.std() + 1e-8)

            e1 = model.encoder(x1)
            e2 = model.encoder(x2)

            dist = torch.norm(e1 - e2).item()

            same = int(labels[a] == labels[b])
            pred = int(dist < threshold)

            if same == 0:
                total_far += 1
                if pred == 1:
                    far += 1

            if same == 1:
                total_frr += 1
                if pred == 0:
                    frr += 1

    print("FAR:", far / total_far)
    print("FRR:", frr / total_frr)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

def plot_roc(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - AES Trojan Detection")
    plt.legend()
    plt.show()